# HomeRadar

Scraper de inmuebles de fincaraiz.com.co con scraping en una sola pasada, reintentos, logging y guardado incremental.

Pipeline:
1. **Scraping** -> `propiedades_raw.csv` (datos crudos tal cual los entrega el sitio)
2. **Limpieza** -> `propiedades_limpias.csv` (numerico, dedupe, valor_m2)
3. **Analisis** -> ranking + percentiles + flag de ganga + graficos

In [ ]:
import csv
import json
import logging
import random
import re
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Iterable, Optional

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------------------------------------------------------
# Configuracion
# ---------------------------------------------------------------------------
BASE_URL = "https://www.fincaraiz.com.co"
START_URL = (
    "https://www.fincaraiz.com.co/venta/apartamentos/santa-paula/zona-norte/"
    "bogota/2-habitaciones/2-banos/1-parqueadero"
)
OUTPUT_RAW = Path("propiedades_raw.csv")
MAX_PAGES = 50               # tope de seguridad
REQUEST_TIMEOUT = 15         # segundos
SLEEP_RANGE = (1.0, 2.5)     # pausa aleatoria entre requests
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36"
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("homeradar")


# ---------------------------------------------------------------------------
# HTTP session con retries
# ---------------------------------------------------------------------------
def build_session() -> requests.Session:
    s = requests.Session()
    retry_kwargs = dict(
        total=4,
        backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        raise_on_status=False,
    )
    # urllib3 >=1.26 usa allowed_methods; versiones antiguas usan method_whitelist
    try:
        retry = Retry(allowed_methods=("GET",), **retry_kwargs)
    except TypeError:
        retry = Retry(method_whitelist=("GET",), **retry_kwargs)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-CO,es;q=0.9"})
    return s


def fetch(session: requests.Session, url: str) -> Optional[BeautifulSoup]:
    try:
        r = session.get(url, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
    except requests.RequestException as exc:
        log.warning("GET fallo %s -> %s", url, exc)
        return None
    return BeautifulSoup(r.text, "html.parser")


# ---------------------------------------------------------------------------
# Modelo y parsing
# ---------------------------------------------------------------------------
FIELDS = [
    "link", "title", "precio", "habitaciones", "banos", "area",
    "estrato", "administracion", "antiguedad", "ubicacion", "parqueaderos",
]


@dataclass
class Property:
    link: str
    title: str = ""
    precio: str = ""
    habitaciones: str = ""
    banos: str = ""
    area: str = ""
    estrato: str = ""
    administracion: str = ""
    antiguedad: str = ""
    ubicacion: str = ""
    parqueaderos: str = ""


# Mapeo tolerante: clave normalizada -> nombre del campo en Property
DETAIL_KEYS = {
    "precio":          "precio",
    "habitaciones":    "habitaciones",
    "banos":           "banos",
    "baños":           "banos",
    "area construida": "area",
    "área construida": "area",
    "area":            "area",
    "área":            "area",
    "estrato":         "estrato",
    "administracion":  "administracion",
    "administración":  "administracion",
    "antiguedad":      "antiguedad",
    "antigüedad":      "antiguedad",
    "ubicacion":       "ubicacion",
    "ubicación":       "ubicacion",
    "parqueaderos":    "parqueaderos",
}


def _norm(text: str) -> str:
    return text.strip().lower()


def get_property_links(soup: BeautifulSoup) -> list[str]:
    cards = soup.select("div.listingCard a.lc-data")
    return [BASE_URL + c["href"] for c in cards if c.get("href")]


def parse_property(soup: BeautifulSoup, url: str) -> Property:
    prop = Property(link=url)

    # Titulo
    if (h1 := soup.find("h1")) is not None:
        prop.title = h1.get_text(strip=True)

    # Precio: probar varios selectores en orden
    for sel in (".property-price-tag", ".price", "span.ant-typography strong"):
        if (node := soup.select_one(sel)) is not None:
            txt = node.get_text(strip=True)
            if txt:
                prop.precio = txt
                break

    # Tipologia (habs / banos / area) cuando aparece junta
    typology = [t.get_text(strip=True) for t in soup.select(".property-typology-tag-desktop span")]
    if len(typology) >= 3:
        prop.habitaciones = prop.habitaciones or typology[0]
        prop.banos        = prop.banos        or typology[1]
        prop.area         = prop.area         or typology[2]

    # Tabla clave/valor de detalle
    for row in soup.select("div.ant-row.ant-row-space-between"):
        spans = row.find_all("span")
        if len(spans) != 2:
            continue
        key = _norm(spans[0].get_text())
        val = spans[1].get_text(strip=True)
        if (field_name := DETAIL_KEYS.get(key)) and not getattr(prop, field_name):
            setattr(prop, field_name, val)

    return prop


# ---------------------------------------------------------------------------
# Guardado incremental
# ---------------------------------------------------------------------------
class CsvSink:
    """Escribe filas al CSV inmediatamente; sobrevive a un crash a mitad de scraping."""

    def __init__(self, path: Path, fieldnames: list[str]):
        self.path = path
        self.fieldnames = fieldnames
        self._fh = path.open("w", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(self._fh, fieldnames=fieldnames)
        self._writer.writeheader()
        self._fh.flush()
        self.count = 0

    def write(self, prop: Property) -> None:
        self._writer.writerow(asdict(prop))
        self._fh.flush()
        self.count += 1

    def close(self) -> None:
        self._fh.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------
def scrape(start_url: str = START_URL, output: Path = OUTPUT_RAW, max_pages: int = MAX_PAGES) -> Path:
    session = build_session()
    seen_links: set[str] = set()

    with CsvSink(output, FIELDS) as sink:
        for page in range(1, max_pages + 1):
            url = f"{start_url}/pagina{page}"
            log.info("Pagina %d -> %s", page, url)
            soup = fetch(session, url)
            if soup is None:
                log.warning("Saltando pagina %d por error de red", page)
                continue

            links = get_property_links(soup)
            if not links:
                log.info("Pagina sin links, fin del scraping")
                break
            new_links = [l for l in links if l not in seen_links]
            if not new_links:
                log.info("Solo duplicados (pagina ya vista), fin del scraping")
                break

            for link in new_links:
                seen_links.add(link)
                time.sleep(random.uniform(*SLEEP_RANGE))
                detail = fetch(session, link)
                if detail is None:
                    continue
                try:
                    prop = parse_property(detail, link)
                    sink.write(prop)
                    log.info("  %s | %s", prop.precio or "?", prop.title[:60])
                except (AttributeError, KeyError) as exc:
                    log.warning("Error parseando %s -> %s", link, exc)

            time.sleep(random.uniform(*SLEEP_RANGE))

        log.info("Listo: %d propiedades en %s", sink.count, output)
    return output


# Ejecutar
scrape()

## Limpieza

Convierte los strings crudos en numericos, deduplica por link y calcula `valor_m2`.

In [ ]:
import re
import pandas as pd

INPUT_FILE = "propiedades_raw.csv"
OUTPUT_FILE = "propiedades_limpias.csv"


def limpiar_precio(precio_str):
    """Extrae el primer numero largo (>=6 digitos con separadores) del texto."""
    if not precio_str or pd.isna(precio_str):
        return None
    # Acepta '$ 450.000.000', 'COP 450,000,000', '450000000', etc.
    match = re.search(r"(\d[\d\.\,]{5,})", str(precio_str))
    if not match:
        return None
    digits = re.sub(r"[^\d]", "", match.group(1))
    return int(digits) if digits else None


def limpiar_area(area_str):
    if not area_str or pd.isna(area_str):
        return None
    # Anclado a unidad de area: m2, m², mts, mts2 (case-insensitive)
    match = re.search(r"([\d\.\,]+)\s*m(?:ts)?[²2]?\b", str(area_str), flags=re.IGNORECASE)
    if not match:
        return None
    val = match.group(1).replace(".", "").replace(",", ".")
    try:
        return float(val)
    except ValueError:
        return None


def limpiar_numero(texto):
    if not texto or pd.isna(texto):
        return None
    match = re.search(r"(\d+)", str(texto))
    return int(match.group(1)) if match else None


df_raw = pd.read_csv(INPUT_FILE)

df_limpio = pd.DataFrame({
    "link":         df_raw["link"],
    "title":        df_raw["title"] if "title" in df_raw.columns else "",
    "precio":       df_raw["precio"].apply(limpiar_precio),
    "habitaciones": df_raw["habitaciones"].apply(limpiar_numero),
    "banos":        df_raw["banos"].apply(limpiar_numero),
    "area_m2":      df_raw["area"].apply(limpiar_area),
})

# Columnas opcionales (solo si existen en el CSV)
if "estrato" in df_raw.columns:
    df_limpio["estrato"] = df_raw["estrato"].apply(limpiar_numero)

# Dedupe por link
df_limpio = df_limpio.drop_duplicates(subset="link").reset_index(drop=True)

# valor_m2 vectorizado
df_limpio["valor_m2"] = (df_limpio["precio"] / df_limpio["area_m2"]).round(2)

# Quitar filas sin precio o sin area (no sirven para el ranking)
df_validas = df_limpio.dropna(subset=["precio", "area_m2"]).copy()

df_limpio.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Total: {len(df_limpio)} | Validas (con precio y area): {len(df_validas)}")
df_limpio.head()

## Analisis

Estadisticas, percentiles y flag de ganga (`valor_m2` por debajo del percentil 25).

In [ ]:
# Estadisticas resumen
stats = df_validas["valor_m2"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("Distribucion valor_m2 (COP/m2):")
print(stats.round(0))

p25 = df_validas["valor_m2"].quantile(0.25)
mediana = df_validas["valor_m2"].median()
df_validas["vs_mediana_pct"] = ((df_validas["valor_m2"] / mediana) - 1) * 100
df_validas["ganga"] = df_validas["valor_m2"] <= p25

print(f"\nMediana valor_m2: {mediana:,.0f} COP/m2")
print(f"Umbral ganga (p25): {p25:,.0f} COP/m2")
print(f"Gangas detectadas: {df_validas['ganga'].sum()} / {len(df_validas)}")

In [ ]:
# Top gangas
df_ordenado = df_validas.sort_values(by="valor_m2", ascending=True)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

cols = ["valor_m2", "precio", "area_m2", "habitaciones", "banos", "vs_mediana_pct", "ganga", "link"]
df_ordenado[cols].head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_validas["valor_m2"].plot.hist(bins=20, ax=axes[0], edgecolor="black")
axes[0].axvline(mediana, color="red", linestyle="--", label=f"mediana {mediana:,.0f}")
axes[0].axvline(p25, color="green", linestyle="--", label=f"p25 {p25:,.0f}")
axes[0].set_title("Distribucion valor_m2")
axes[0].set_xlabel("COP / m2")
axes[0].legend()

df_validas.plot.scatter(x="area_m2", y="precio", ax=axes[1], alpha=0.6)
axes[1].set_title("Precio vs Area")
axes[1].set_xlabel("m2")
axes[1].set_ylabel("COP")

plt.tight_layout()
plt.show()